## Trying out compiling the information as a tensor ##

In [ ]:
from lidar_object_tracking.dataset import load_json_data
import numpy as np
import plotly.graph_objects as go
from lidar_object_tracking.config import PROCESSED_DATA_DIR

2026-03-03 12:40:14.283 | INFO     | lidar_object_tracking.config:<module>:11 - PROJ_ROOT path is: /home/danielb/Desktop/CodingProjects/lidar_object_tracking


In [15]:
def json_to_numpy(directory,scene_num):
    data_dict = load_json_data(directory,scene_num)
    total_points = len(data_dict)*len(data_dict[0])

    i = 0
    dataset = np.zeros((total_points,4))
    for key in range(80):
        numpy_data = data_dict[key].to_numpy()
        for datum in numpy_data:
            frame_datum = np.insert(datum, 0, key*0.1)
            dataset[i] = frame_datum
            i += 1
    return dataset
def pull_data_with_labels(labeled_dataset,labels, label):
    if label not in labels:
        raise ValueError(f'{label} is not a valid label for the dataset.')
    
    filt = (labels == label)

    data_with_label = labeled_dataset[filt]

    label_dict = {}
    for value in np.unique(data_with_label[:,0]):
        label_dict[int(value*10)] = data_with_label[data_with_label[:,0] == value][:,[1,2,3]]
    return dict(sorted(label_dict.items()))

def fill_in_frames(dataset_dict:dict):
    key_list = sorted(dataset_dict.keys())

    for idx in range(min(key_list), max(key_list)+1):
        try:
            dataset_dict[idx]
        except KeyError:
            dataset_dict[idx] = dataset_dict[idx - 1]

    return dict(sorted(dataset_dict.items()))    

def plot_with_slider(dataset:dict):
    aug_dataset = {}
    for key in dataset.keys():
        aug_dataset[key] = np.concatenate((dataset[key], [[-10,0,0],[10,15,5]]))

    fig = go.Figure()

    max_frame = max(aug_dataset.keys())
    min_frame = min(aug_dataset.keys())
    span_frame = (max_frame - min_frame)
    for i in range(min_frame, max_frame+1):

        fig.add_trace(
            go.Scatter3d(
                visible=False,
                mode="markers",
                marker=dict(size=2),
                x=aug_dataset[i][:,0],
                y=aug_dataset[i][:,1],
                z=aug_dataset[i][:,2],
            )
        )
        fig.update_layout(height=500, width=750)
        fig.update_scenes(
            xaxis_range=[-10, 10],
            yaxis_range=[0, 15],
            zaxis_range=[0, 5],
            aspectmode="data",
            overwrite=True
        )

    fig.data[0].visible = True

    steps = []
    for i in range(len(list(fig.data))):
        step = dict(
            method="update",
            args=[{"visible": [False] * len(list(fig.data))}],
            label=f"{i / 10}s",
        )
        step["args"][0]["visible"][i] = True
        steps.append(step)

    sliders = [
        dict(
            active=0,
            currentvalue={
                "prefix": "Frame: ",
            },
            pad={"t": 80},
            steps=steps,
        )
    ]

    fig.update_layout(sliders=sliders)

    fig.show()


In [17]:
dataset = json_to_numpy(PROCESSED_DATA_DIR,'004')

In [18]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps = 1, min_samples = 50)

clusters = db.fit(dataset)

labels = clusters.labels_

np.unique_counts(labels)

UniqueCountsResult(values=array([-1,  0,  1,  2,  3,  4,  5]), counts=array([  125,  9464, 10047,  8868, 11225,   175,    96]))

In [19]:
processed_labeled_dataset = fill_in_frames(pull_data_with_labels(dataset,labels, label = 1))

plot_with_slider(processed_labeled_dataset)